# Análise Exploratória do Mercado de Cinema Brasileiro

Análise exploratória dos dados de bilheteria do mercado cinematográfico nacional,
utilizando dados públicos da **ANCINE (Agência Nacional do Cinema)**.

**Período coberto:** janeiro de 2014 a junho de 2026

> ⚠️ Os dados de **2026 são parciais** — cobrem apenas de janeiro a junho.
> Todas as comparações anuais que incluam 2026 indicarão isso explicitamente.

---

## Perguntas investigadas

1. **Impacto da pandemia de COVID-19:** Como e quando o mercado foi afetado?
   Qual a magnitude da queda?
2. **Recuperação do mercado:** O público retornou aos patamares pré-pandemia?
   Em que ritmo?
3. **Filmes premiados:** Qual o impacto de *Ainda Estou Aqui* e *O Agente Secreto*
   na bilheteria do cinema nacional?
4. **Cinema nacional vs. estrangeiro:** Como evolui a participação do cinema
   brasileiro ao longo dos anos?
5. **Sazonalidade:** Existem padrões sazonais consistentes no consumo de cinema?
6. **Distribuição geográfica:** Como o público se distribui entre estados e regiões?

---

> **Sobre os dados:** Os registros da ANCINE contabilizam **público** (número de
> espectadores por sessão/dia/sala), não receita financeira. A identificação de filmes
> nacionais é feita pelo prefixo `B` no campo `CPB_ROE` (código de registro ANCINE).

---
## Seção 0 — Setup e Configurações

Nesta seção preparamos o ambiente de análise:
- Importação das bibliotecas necessárias
- Definição dos caminhos de entrada e saída
- Configurações visuais do matplotlib
- Funções auxiliares para salvar figuras e formatar eixos

In [1]:
# ── Manipulação e análise de dados ──────────────────────────
import pandas as pd
import numpy as np

# ── Visualização ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Utilitários de sistema ───────────────────────────────────
import glob
import warnings
from pathlib import Path

# Suprimir avisos não críticos para manter o output do notebook limpo
warnings.filterwarnings('ignore')

# Confirmar versões das principais bibliotecas
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')
print(f'seaborn {sns.__version__}')

pandas  2.3.3
numpy   2.0.2
seaborn 0.13.2


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CAMINHOS
# ═══════════════════════════════════════════════════════════════
# O notebook fica em notebooks/, então o diretório raiz está
# um nível acima. Path.resolve() garante o caminho absoluto.
ROOT_DIR = Path('..').resolve()

DATA_DIR        = ROOT_DIR / 'data'                 # CSVs brutos ANCINE (não versionados)
FIGURES_DIR     = ROOT_DIR / 'outputs' / 'figures'  # Gráficos exportados (versionados)
OUTPUT_DATA_DIR = ROOT_DIR / 'outputs' / 'processados'  # Dados processados/agregados (versionados)

# Criar os diretórios de saída caso não existam
# (útil quando o notebook é executado em um ambiente limpo)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dados brutos:   {DATA_DIR}')
print(f'Figuras:        {FIGURES_DIR}')
print(f'Dados gerados:  {OUTPUT_DATA_DIR}')

# ═══════════════════════════════════════════════════════════════
# CONFIGURAÇÕES DE VISUALIZAÇÃO
# ═══════════════════════════════════════════════════════════════
# Estilo base: grade de fundo clara, sem bordas superiores/direitas
plt.style.use('seaborn-v0_8-whitegrid')

# Parâmetros globais aplicados a todos os gráficos do notebook
plt.rcParams.update({
    'figure.figsize':    (14, 6),  # tamanho padrão: largo e não muito alto
    'figure.dpi':        100,
    'font.size':         12,
    'axes.titlesize':    14,
    'axes.labelsize':    12,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   11,
    'axes.spines.top':   False,  # remover bordas superior e direita
    'axes.spines.right': False,  # deixa o gráfico mais limpo visualmente
})

# Paleta de cores padronizada para o projeto.
# Usar sempre as mesmas cores para os mesmos conceitos facilita
# a leitura e a comparação entre gráficos.
CORES = {
    'nacional':    '#1f77b4',  # azul    — cinema nacional
    'estrangeiro': '#ff7f0e',  # laranja — cinema estrangeiro
    'pandemia':    '#d62728',  # vermelho — período pandemia
    'destaque':    '#2ca02c',  # verde   — filmes premiados
    'neutro':      '#7f7f7f',  # cinza   — 2026 (ano incompleto)
    'total':       '#17becf',  # ciano   — totais gerais
}

# ═══════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES
# ═══════════════════════════════════════════════════════════════
def salvar_figura(nome: str) -> None:
    """Salva a figura matplotlib atual em outputs/figures/."""
    caminho = FIGURES_DIR / nome
    plt.savefig(caminho, dpi=150, bbox_inches='tight')
    print(f'Figura salva: outputs/figures/{nome}')


def formatar_milhoes(x, pos=None) -> str:
    """Formata valores do eixo Y em milhões (ex: 5,2M) para legibilidade."""
    return f'{x/1_000_000:.1f}M'


print('Configurações aplicadas com sucesso.')

---
## Seção 1 — Carregamento e Otimização de Memória

Os dados da ANCINE estão distribuídos em **150 arquivos CSV** mensais
(jan/2014 a jun/2026), totalizando aproximadamente **4,5 GB em disco** e
**20,9 milhões de linhas**.

### Estratégia de carregamento

Carregar tudo de uma vez exige cuidado com o uso de memória. As otimizações adotadas são:

| Técnica | Por quê | Ganho estimado |
|---------|---------|----------------|
| `dtype='category'` para colunas com poucos valores únicos | Armazena como inteiro com tabela de mapeamento | Até 90% vs. `object` |
| `dtype='int32'` para `PUBLICO` | Valores até ~2.100 — não precisa de int64 | 50% vs. `int64` |
| `dtype='Int32'` (nullable) para IDs com nulos | Evita conversão automática para `float64` | 50% vs. `float64` |
| Descartar 3 colunas não utilizadas na leitura | Reduz a quantidade de dados carregados | ~15% |
| Converter `DATA_EXIBICAO` após o concat | Parsear datas uma vez só, no DataFrame completo | Mais rápido |

**Resultado esperado:** ~1,5–2,0 GB em RAM (vs. ~4,5 GB sem otimização de tipos).

In [3]:
# ═══════════════════════════════════════════════════════════════
# TIPOS DE DADOS OTIMIZADOS
# ═══════════════════════════════════════════════════════════════
# Definir os dtypes NA LEITURA (e não depois) é essencial:
# se carregarmos como object/int64 e convertermos depois, o pandas
# mantém as duas versões na memória simultaneamente durante a conversão.

DTYPES = {
    # ── Texto com alta repetição → 'category' ─────────────────
    # 'category' armazena os dados como inteiros internamente,
    # mantendo apenas uma tabela de mapeamento para os valores únicos.
    # Ideal quando os mesmos valores se repetem milhões de vezes.

    'TITULO_ORIGINAL':            'category',  # mesmo filme em N salas × N dias
    'TITULO_BRASIL':              'category',  # mesma lógica
    'CPB_ROE':                    'category',  # código ANCINE — B=nacional, E=estrangeiro
    'PAIS_OBRA':                  'category',  # ~50 países únicos em todo o dataset
    'NOME_SALA':                  'category',  # cada sala repete todos os dias que exibe
    'MUNICIPIO_SALA_COMPLEXO':    'category',  # ~500 municípios únicos
    'UF_SALA_COMPLEXO':           'category',  # exatamente 27 valores (siglas dos estados)
    'RAZAO_SOCIAL_DISTRIBUIDORA': 'category',  # poucas dezenas de distribuidoras

    # ── Público: int32 é suficiente ───────────────────────────
    # Valores típicos: 1–1.600 espectadores por sala por dia.
    # int32 suporta até ~2,1 bilhões — mais que suficiente.
    'PUBLICO': 'int32',

    # ── IDs numéricos com nulos ───────────────────────────────
    # Int32 (maiúsculo) é a versão nullable do pandas (>=1.0).
    # Sem ele, o pandas converteria automaticamente para float64
    # na presença de qualquer nulo, dobrando o consumo de memória.
    'REGISTRO_SALA':           'Int32',  # 0,16% de nulos em anos antigos
    'REGISTRO_COMPLEXO':       'Int32',
    'REGISTRO_EXIBIDOR':       'Int32',
    'REGISTRO_GRUPO_EXIBIDOR': 'Int32',  # 8,6% de nulos
}

# Colunas sem utilidade para a análise — descartadas durante a leitura
COLUNAS_DESCARTAR = [
    'NR_PROTOCOLO_ENVIO',         # número interno do protocolo ANCINE
    'DATA_HORA_ENVIO_PROTOCOLO',  # timestamp do envio pelo exibidor (≠ data de exibição)
    'CNPJ_DISTRIBUIDORA',         # redundante — já temos RAZAO_SOCIAL_DISTRIBUIDORA
]

print(f'Dtypes otimizados para {len(DTYPES)} colunas')
print(f'Colunas descartadas na leitura: {COLUNAS_DESCARTAR}')

Dtypes otimizados para 13 colunas
Colunas descartadas na leitura: ['NR_PROTOCOLO_ENVIO', 'DATA_HORA_ENVIO_PROTOCOLO', 'CNPJ_DISTRIBUIDORA']


In [4]:
# ═══════════════════════════════════════════════════════════════
# CARREGAMENTO DOS ARQUIVOS CSV
# ═══════════════════════════════════════════════════════════════
# Localizar todos os arquivos no diretório de dados.
# sorted() garante leitura em ordem cronológica (jan/2014 → jun/2026).
arquivos = sorted(glob.glob(str(DATA_DIR / '*.csv')))

print(f'Arquivos encontrados: {len(arquivos)}')
print(f'  Primeiro: {Path(arquivos[0]).name}')
print(f'  Último:   {Path(arquivos[-1]).name}')
print()

# ── Carregamento em lista + concat ao final ───────────────────
# Por que não concat em loop?
# pd.concat dentro de um loop é O(n²) — cada iteração cria uma
# cópia completa do DataFrame acumulado. Com 150 arquivos, isso
# seria extremamente lento e desperdiçaria muita memória.
# A abordagem correta: guardar tudo em lista e fazer um único concat.
print('Carregando arquivos... (pode levar alguns minutos)')

dfs = []
for i, caminho in enumerate(arquivos, 1):
    df_mes = pd.read_csv(
        caminho,
        sep=';',                                        # separador padrão ANCINE
        encoding='utf-8',                              # verificado em todos os 150 arquivos
        dtype=DTYPES,                                  # tipos otimizados definidos acima
        usecols=lambda c: c not in COLUNAS_DESCARTAR, # descartar na leitura, não depois
    )
    dfs.append(df_mes)

    # Exibir progresso a cada 10 arquivos e ao final
    if i % 10 == 0 or i == len(arquivos):
        print(f'  {i:>3}/{len(arquivos)} arquivos carregados...')

# Concatenar tudo em um único DataFrame
# ignore_index=True: reconstrói o índice de 0 a N (evita índices duplicados)
df = pd.concat(dfs, ignore_index=True)

# Liberar a lista intermediária para recuperar a memória alocada
del dfs

print(f'\nCarregamento concluído!')
print(f'  Linhas totais: {len(df):>12,}')
print(f'  Colunas:       {df.shape[1]:>12}')

Arquivos encontrados: 150
  Primeiro: bilheteria-diaria-obras-por-distribuidoras-2014-01.csv
  Último:   bilheteria-diaria-obras-por-distribuidoras-2026-06.csv

Carregando arquivos... (pode levar alguns minutos)
   10/150 arquivos carregados...
   20/150 arquivos carregados...
   30/150 arquivos carregados...
   40/150 arquivos carregados...
   50/150 arquivos carregados...
   60/150 arquivos carregados...
   70/150 arquivos carregados...
   80/150 arquivos carregados...
   90/150 arquivos carregados...
  100/150 arquivos carregados...
  110/150 arquivos carregados...
  120/150 arquivos carregados...
  130/150 arquivos carregados...
  140/150 arquivos carregados...
  150/150 arquivos carregados...

Carregamento concluído!
  Linhas totais:   20,916,062
  Colunas:                 14


In [5]:
# ═══════════════════════════════════════════════════════════════
# CONVERSÃO DA COLUNA DE DATA
# ═══════════════════════════════════════════════════════════════
# DATA_EXIBICAO foi lida como string no formato DD/MM/AAAA.
# Convertemos para datetime APÓS o concat porque é mais eficiente:
# parsear datas uma vez no DataFrame completo é mais rápido do que
# parsear em cada um dos 150 arquivos individualmente.
#
# O formato explícito '%d/%m/%Y' é mais rápido do que inferência automática.
df['DATA_EXIBICAO'] = pd.to_datetime(df['DATA_EXIBICAO'], format='%d/%m/%Y')

# Confirmar o período coberto
data_inicio = df['DATA_EXIBICAO'].min().date()
data_fim    = df['DATA_EXIBICAO'].max().date()
print(f'Período coberto: {data_inicio}  →  {data_fim}')
print()

# ═══════════════════════════════════════════════════════════════
# DIAGNÓSTICO DE MEMÓRIA
# ═══════════════════════════════════════════════════════════════
# memory_usage(deep=True) considera o conteúdo real das colunas
# de tipo object/category (não apenas os ponteiros), dando um
# número preciso do consumo real em memória.
mem_atual_mb = df.memory_usage(deep=True).sum() / 1_048_576

# Tamanho total em disco como referência de comparação
tamanho_disco_mb = sum(
    Path(f).stat().st_size for f in arquivos
) / 1_048_576

print('Uso de memória do DataFrame:')
print(f'  Em memória (otimizado): {mem_atual_mb:>8.0f} MB')
print(f'  Em disco (referência):  {tamanho_disco_mb:>8.0f} MB')
print()

# Detalhe por coluna — identifica as que mais consomem memória
mem_por_coluna = (
    df.memory_usage(deep=True)
      .drop('Index')              # excluir o índice do ranking
      .sort_values(ascending=False)
      .head(8)                    # top 8 colunas mais pesadas
      .apply(lambda x: f'{x/1_048_576:.1f} MB')
)
print('Top 8 colunas por consumo de memória:')
print(mem_por_coluna.to_string())

Período coberto: 2014-01-01  →  2026-06-13

Uso de memória do DataFrame:
  Em memória (otimizado):    13300 MB
  Em disco (referência):      4637 MB

Top 8 colunas por consumo de memória:
RAZAO_SOCIAL_DISTRIBUIDORA    1873.8 MB
NOME_SALA                     1862.6 MB
TITULO_BRASIL                 1741.2 MB
MUNICIPIO_SALA_COMPLEXO       1623.2 MB
TITULO_ORIGINAL               1564.8 MB
PAIS_OBRA                     1416.8 MB
CPB_ROE                       1416.2 MB
UF_SALA_COMPLEXO              1163.4 MB
